# 03 — Fine-tuning CNN ResNet18 (Dual Head)
**Head 1:** Estado de conservación (5 clases)  
**Head 2:** Tipo de ambiente (8 clases)  

El CSV de DL contiene metadatos de imágenes. Para entrenamiento real se deben
tener las imágenes en disco. Este notebook entrena con datos sintéticos de imagen
(ruido RGB) como demostración del pipeline, listo para reemplazar con imágenes reales.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import pandas as pd
import numpy as np
import mlflow
from pathlib import Path
from PIL import Image

CSV_PATH   = '../ml_models/datasets/DL_Imagenes_Propiedades.csv'
MODEL_PATH = '../ml_models/trained/cnn_property.pt'

CONSERVATION_LABELS = ['Excelente','Bueno','Regular','Malo','En construcción']
ROOM_LABELS = ['Sala','Dormitorio','Cocina','Baño','Fachada exterior',
               'Jardín/Patio','Garaje','Otro']

df = pd.read_csv(CSV_PATH)
print(df.shape)
print('Conservación:', df['label_estado_conservacion'].value_counts().to_dict())
print('Ambiente:',     df['label_ambiente'].value_counts().to_dict())


In [ ]:
# ── Dataset sintético (reemplazar con imágenes reales) ────────────────────
class PropertyDataset(Dataset):
    def __init__(self, df, transform, conservation_labels, room_labels):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.cons_map = {v:i for i,v in enumerate(conservation_labels)}
        self.room_map = {v:i for i,v in enumerate(room_labels)}
        # Normalizar etiquetas de ambiente al set de 8 clases
        self.df['room_norm'] = self.df['label_ambiente'].apply(
            lambda x: x if x in self.room_map else 'Otro')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Imagen sintética 224x224 (RGB ruido) — reemplazar con Image.open(path)
        img_array = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
        img = Image.fromarray(img_array)
        x = self.transform(img)
        y_cons = self.cons_map.get(row['label_estado_conservacion'], 0)
        y_room = self.room_map.get(row['room_norm'], 7)
        return x, torch.tensor(y_cons), torch.tensor(y_room)

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

df_train = df[df['split']=='train'].copy()
df_val   = df[df['split']=='val'].copy()

train_ds = PropertyDataset(df_train, transform,     CONSERVATION_LABELS, ROOM_LABELS)
val_ds   = PropertyDataset(df_val,   transform_val, CONSERVATION_LABELS, ROOM_LABELS)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')


In [ ]:
# ── Arquitectura Dual Head ────────────────────────────────────────────────
class PropertyDualHeadCNN(nn.Module):
    def __init__(self, num_conservation=5, num_rooms=8):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_features   = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.head_conservation = nn.Linear(in_features, num_conservation)
        self.head_rooms        = nn.Linear(in_features, num_rooms)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head_conservation(feats), self.head_rooms(feats)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
model = PropertyDualHeadCNN().to(device)


In [ ]:
# ── Entrenamiento ─────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
EPOCHS = 10

mlflow.set_experiment('cnn_propiedad')
with mlflow.start_run(run_name='resnet18_dual_head_v1'):
    mlflow.log_params({'epochs':EPOCHS,'lr':1e-4,'batch_size':32,'backbone':'resnet18'})

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for x, y_cons, y_room in train_dl:
            x, y_cons, y_room = x.to(device), y_cons.to(device), y_room.to(device)
            optimizer.zero_grad()
            out_cons, out_room = model(x)
            loss = criterion(out_cons, y_cons) + criterion(out_room, y_room)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        # Validación
        model.eval()
        correct_cons = correct_room = total = 0
        with torch.no_grad():
            for x, y_cons, y_room in val_dl:
                x, y_cons, y_room = x.to(device), y_cons.to(device), y_room.to(device)
                out_cons, out_room = model(x)
                correct_cons += (out_cons.argmax(1)==y_cons).sum().item()
                correct_room += (out_room.argmax(1)==y_room).sum().item()
                total += len(y_cons)
        acc_cons = correct_cons/total
        acc_room = correct_room/total
        avg_loss = total_loss/len(train_dl)
        mlflow.log_metrics({'loss':avg_loss,'acc_cons':acc_cons,'acc_room':acc_room}, step=epoch)
        print(f'Epoch {epoch+1}/{EPOCHS} | Loss:{avg_loss:.4f} | Acc cons:{acc_cons:.3f} | Acc room:{acc_room:.3f}')


In [ ]:
# ── Guardar modelo ────────────────────────────────────────────────────────
Path('../ml_models/trained').mkdir(parents=True, exist_ok=True)
torch.save({
    'backbone':          model.backbone.state_dict(),
    'head_conservation': model.head_conservation.state_dict(),
    'head_rooms':        model.head_rooms.state_dict(),
    'conservation_labels': CONSERVATION_LABELS,
    'room_labels':         ROOM_LABELS,
}, MODEL_PATH)
print(f'Modelo guardado en: {MODEL_PATH}')

# Test inferencia
model.eval()
import torch.nn.functional as F
test_img = torch.randn(1,3,224,224).to(device)
with torch.no_grad():
    out_c, out_r = model(test_img)
    prob_c = F.softmax(out_c,dim=1).squeeze().cpu().numpy()
    prob_r = F.softmax(out_r,dim=1).squeeze().cpu().numpy()
print('Estado:', CONSERVATION_LABELS[prob_c.argmax()], f'({prob_c.max():.2%})')
print('Ambiente:', ROOM_LABELS[prob_r.argmax()], f'({prob_r.max():.2%})')
